## Example 1: a slider keeps its value inside its range

In [ ]:
import ipywidgets as widgets

slider = widgets.IntSlider(value=3, min=0, max=10, description="n")
print(f"{slider.description}: {slider.value} in [{slider.min}, {slider.max}]")
assert slider.value == 3

slider.value = 7
assert slider.value == 7
slider.value = 99  # above max: clamped
assert slider.value == 10, slider.value
slider.value = -5  # below min: clamped
assert slider.value == 0, slider.value

ratio = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.1)
ratio.value = 2.0
assert ratio.value == 1.0, ratio.value
print("values outside the range are clamped to min and max")

## Example 2: linking two widgets

In [ ]:
import ipywidgets as widgets

a = widgets.IntSlider(value=1, min=0, max=10)
b = widgets.IntText(value=0)

link = widgets.link((a, "value"), (b, "value"))  # two-way
assert b.value == 1  # the target takes the source value straight away
a.value = 4
assert b.value == 4
b.value = 6
assert a.value == 6
link.unlink()
a.value = 2
assert b.value == 6  # no longer linked

c = widgets.IntText(value=0)
one_way = widgets.dlink((a, "value"), (c, "value"), transform=lambda v: v * 10)
a.value = 3
assert c.value == 30
c.value = 99
assert a.value == 3  # dlink only goes one way
one_way.unlink()
print("link() is two-way, dlink() one-way with an optional transform")

## Example 3: containers and their children

In [ ]:
import ipywidgets as widgets

a = widgets.IntSlider()
b = widgets.Text(value="hello")
column = widgets.VBox(children=[a, b])
row = widgets.HBox(children=[column, widgets.Button(description="Go")])
assert column.children == (a, b)
assert row.children[0] is column and row.children[1].description == "Go"

tabs = widgets.Tab(children=[widgets.Label("first"), widgets.Label("second")])
tabs.titles = ("one", "two")
tabs.selected_index = 1
assert tabs.titles == ("one", "two") and tabs.selected_index == 1

accordion = widgets.Accordion(children=[widgets.Label("x")], titles=("Section",))
assert accordion.titles == ("Section",)
print(f"VBox with {len(column.children)} children, Tab titles {tabs.titles}")

## Example 4: observing changes and validating values

In [ ]:
import ipywidgets as widgets
from traitlets import TraitError

slider = widgets.IntSlider(value=1, min=0, max=10)
seen = []
slider.observe(lambda change: seen.append((change["old"], change["new"])), names="value")
slider.value = 5
slider.value = 8
print(f"changes seen: {seen}")
assert seen == [(1, 5), (5, 8)]

colour = widgets.Dropdown(options=["red", "green", "blue"], value="green")
assert colour.index == 1
colour.index = 2
assert colour.value == "blue"
try:
    colour.value = "purple"
except TraitError as error:
    print(f"rejected a value that is not an option: {error}")
else:
    raise AssertionError("Dropdown accepted a value outside its options")

text = widgets.Text(value="hi")
assert text.get_state()["value"] == "hi"  # the state that is synced to the front end

## Example 5: `interactive` builds controls from a function's arguments

In [ ]:
import ipywidgets as widgets

def double(x):
    return x * 2

panel = widgets.interactive(double, x=5)
controls = [type(child).__name__ for child in panel.children]
print(f"controls: {controls}, result: {panel.result}")
assert isinstance(panel.children[0], widgets.IntSlider)
assert panel.result == 10

panel.children[0].value = 7  # moving the slider calls the function again
assert panel.result == 14

choice = widgets.interactive(lambda word: word.upper(), word=["low", "high"])
assert isinstance(choice.children[0], widgets.Dropdown)
assert choice.result == "LOW"
print("interactive() picked an IntSlider for an int and a Dropdown for a list")